# 01 — LoRA Fine-tune Runner (Strict Split) — **V2**
Sesuai diagram kamu:
- **Phase 1:** tuning **LR** (rank LoRA **r tetap**)
- **Phase 2:** tuning **rank r** (LR **tetap**)
- **Retrain final:** train\_strict + val\_strict (tanpa tuning lagi) lalu **evaluasi sekali** di test\_strict

Tambahan V2:
- ✅ **Sanity check** (1-step train + 1-step val)
- ✅ **Auto-pick batch size** (coba beberapa BATCH\_SIZE, ambil yang paling besar tanpa OOM)
- ✅ Metrik lengkap: **MAE, (1−MAE), RMSE, R2** (mean 5 trait + per trait)


In [ ]:
import os, json, math, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import soundfile as sf
from transformers import AutoFeatureExtractor, WavLMModel
from peft import LoraConfig, TaskType, get_peft_model
from torch.amp import autocast, GradScaler

# optional: reduce CUDA fragmentation
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


In [ ]:
# ===== PATHS (match notebook 00)
ROOT = Path("/workspace/ta_finetune")          # EDIT kalau beda
DATA = ROOT / "data"
OUT_ROOT = ROOT / "outputs" / "finetune_strict_wavlm_lora_v2"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST = DATA / "manifest_strict_vast.csv"   # dari notebook 00 (strict split + audio_path)
assert MANIFEST.exists(), f"Missing: {MANIFEST}. Jalankan notebook 00 dulu."

df = pd.read_csv(MANIFEST)

SPLIT_COL = "split_strict"
AUDIO_PATH_COL = "audio_path"
LABEL_COLS = ["extraversion","neuroticism","agreeableness","conscientiousness","openness"]

for c in [SPLIT_COL, AUDIO_PATH_COL] + LABEL_COLS:
    assert c in df.columns, f"Missing col: {c}"

df_train = df[df[SPLIT_COL]=="train"].copy()
df_val   = df[df[SPLIT_COL]=="val"].copy()
df_test  = df[df[SPLIT_COL]=="test"].copy()

print("train/val/test:", len(df_train), len(df_val), len(df_test))


In [ ]:
# ===== SEED
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# ===== MODEL / AUDIO
MODEL_NAME = "microsoft/wavlm-base-plus"
SR_TARGET = 16000
MAX_SEC = 15.0
MAX_LEN = int(SR_TARGET * MAX_SEC)

# ===== TRAINING DEFAULTS (sesuai diagram)
WEIGHT_DECAY = 0.01
MAX_EPOCH = 20
PATIENCE = 5
GRAD_CLIP = 1.0
USE_AMP = True

# ===== DATALOADER
BATCH_SIZE = 2          # akan dioverride kalau AUTO_BATCH=True
NUM_WORKERS = 0         # kalau sudah stabil, bisa 2

# ===== CAPACITY HELPERS
AUTO_BATCH = True
BS_CANDIDATES = [2, 4, 6, 8, 10, 12, 16, 20]  # coba naik bertahap

# (opsional) kalau kamu pengen "effective batch" besar tanpa OOM:
TARGET_EFFECTIVE_BS = None   # contoh: 20  -> otomatis set grad_accum_steps


In [ ]:
feat = AutoFeatureExtractor.from_pretrained(MODEL_NAME)

def _trim_pad(wav: np.ndarray):
    if wav.shape[0] > MAX_LEN:
        return wav[:MAX_LEN]
    if wav.shape[0] < MAX_LEN:
        return np.pad(wav, (0, MAX_LEN - wav.shape[0]), mode="constant")
    return wav

class StrictAudioDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = Path(row[AUDIO_PATH_COL])
        wav, sr = sf.read(p)
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        # idealnya audio kamu sudah 16k dari preprocessing, tapi ini fallback aman:
        if sr != SR_TARGET:
            import librosa
            wav = librosa.resample(wav.astype(np.float32), orig_sr=sr, target_sr=SR_TARGET)
        wav = _trim_pad(wav.astype(np.float32))

        y = row[LABEL_COLS].to_numpy(dtype=np.float32)  # (5,)
        return wav, y

def collate_fn(batch):
    wavs, ys = zip(*batch)
    inputs = feat(list(wavs), sampling_rate=SR_TARGET, return_tensors="pt", padding=True)
    y = torch.tensor(np.stack(ys), dtype=torch.float32)
    return inputs, y

def make_loaders(df_tr, df_va, batch_size: int):
    tr_ds = StrictAudioDataset(df_tr)
    va_ds = StrictAudioDataset(df_va)
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"),
                       persistent_workers=False, collate_fn=collate_fn)
    va_dl = DataLoader(va_ds, batch_size=batch_size, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"),
                       persistent_workers=False, collate_fn=collate_fn)
    return tr_dl, va_dl


In [ ]:
class WavLMWithHead(nn.Module):
    def __init__(self, backbone_name: str, r: int, lora_alpha: int = 32, lora_dropout: float = 0.05):
        super().__init__()
        base = WavLMModel.from_pretrained(backbone_name)

        # freeze backbone asli (sesuai diagram)
        for p in base.parameters():
            p.requires_grad = False

        # LoRA on q_proj & v_proj
        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=r,
            lora_alpha=lora_alpha,
            lora_dropout=lora_dropout,
            target_modules=["q_proj", "v_proj"],
            bias="none",
        )
        self.backbone = get_peft_model(base, lora_cfg)

        hidden = base.config.hidden_size
        self.head = nn.Sequential(
            nn.Linear(hidden, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, 5),
            nn.Sigmoid(),  # output 0..1 biar match label
        )

    def mean_pool(self, x, attn_mask=None):
        # x: (B,T,H), attn_mask: (B,T)
        if attn_mask is None:
            return x.mean(dim=1)
        m = attn_mask.unsqueeze(-1).type_as(x)
        return (x * m).sum(dim=1) / (m.sum(dim=1).clamp(min=1.0))

    def forward(self, input_values, attention_mask=None):
        out = self.backbone(input_values=input_values, attention_mask=attention_mask)
        h = out.last_hidden_state  # (B, T_feat, H)

        feat_mask = None
        if attention_mask is not None:
            # ambil base model (karena backbone sudah di-wrap PEFT)
            base = self.backbone.base_model if hasattr(self.backbone, "base_model") else self.backbone
            if hasattr(base, "_get_feature_vector_attention_mask"):
                feat_mask = base._get_feature_vector_attention_mask(h.shape[1], attention_mask)

        pooled = self.mean_pool(h, feat_mask)
        return self.head(pooled)

def trainable_params(model: nn.Module):
    return [p for p in model.parameters() if p.requires_grad]

def count_params(model: nn.Module):
    tot = sum(p.numel() for p in model.parameters())
    trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, trn


In [ ]:
def eval_metrics(model, dl):
    model.eval()
    ys, yhats = [], []
    with torch.no_grad():
        for inputs, y in dl:    
            iv = inputs["input_values"].to(DEVICE, non_blocking=True)
            am = inputs.get("attention_mask", None)
            if am is not None:
                am = am.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast(device_type=("cuda" if DEVICE=="cuda" else "cpu"), enabled=(USE_AMP and DEVICE=="cuda")):
                yhat = model(iv, am)

            ys.append(y.detach().cpu())
            yhats.append(yhat.detach().cpu())

    y_true = torch.cat(ys, dim=0).numpy()    # (N,5)
    y_pred = torch.cat(yhats, dim=0).numpy() # (N,5)

    # MAE
    mae_per = np.mean(np.abs(y_pred - y_true), axis=0)     # (5,)
    mae_mean = float(np.mean(mae_per))

    # RMSE
    rmse_per = np.sqrt(np.mean((y_pred - y_true)**2, axis=0))
    rmse_mean = float(np.mean(rmse_per))

    # R2 per trait: 1 - SSE/SST (handle SST=0)
    r2_per = []
    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_pred[:, j]
        sse = float(np.sum((yt - yp)**2))
        sst = float(np.sum((yt - np.mean(yt))**2))
        r2 = 1.0 - (sse / sst) if sst > 1e-12 else np.nan
        r2_per.append(r2)
    r2_per = np.array(r2_per, dtype=float)
    r2_mean = float(np.nanmean(r2_per))

    # 1 - MAE
    acc_per = 1.0 - mae_per
    acc_mean = float(np.mean(acc_per))

    # skor seleksi (diagram kamu): S = 1 - MAE_mean
    S = 1.0 - mae_mean

    return {
        "mae_per": mae_per, "mae_mean": mae_mean,
        "acc_per": acc_per, "acc_mean": acc_mean,  # 1-MAE
        "rmse_per": rmse_per, "rmse_mean": rmse_mean,
        "r2_per": r2_per, "r2_mean": r2_mean,
        "S": S,
    }


In [ ]:
from tqdm.auto import tqdm

def train_one_epoch(model, dl, opt, scaler, grad_accum_steps: int = 1):
    model.train()
    total = 0.0
    n = 0
    opt.zero_grad(set_to_none=True)

    step = 0
    pbar = tqdm(enumerate(dl, start=1), total=len(dl), desc="train", leave=False)

    for step, (inputs, y) in pbar:
        iv = inputs["input_values"].to(DEVICE, non_blocking=True)
        am = inputs.get("attention_mask", None)
        if am is not None:
            am = am.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with autocast(device_type=("cuda" if DEVICE=="cuda" else "cpu"),
                      enabled=(USE_AMP and DEVICE=="cuda")):
            yhat = model(iv, am)
            loss = F.l1_loss(yhat, y, reduction="mean")

        loss_scaled = loss / max(grad_accum_steps, 1)
        scaler.scale(loss_scaled).backward()

        if (step % grad_accum_steps) == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable_params(model), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)

        bs = y.size(0)
        total += float(loss.item()) * bs
        n += int(bs)

        pbar.set_postfix(loss=float(loss.item()), avg=float(total/max(n,1)))

    if (step % grad_accum_steps) != 0:
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(trainable_params(model), GRAD_CLIP)
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)

    return total / max(n, 1)


In [ ]:
def save_checkpoint(run_dir: Path, model, opt, epoch: int, best_S: float, cfg: dict, best_metrics: dict):
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = run_dir / "checkpoint_best.pt"
    torch.save({
        "epoch": epoch,
        "best_S": best_S,
        "model_state": model.state_dict(),
        "opt_state": opt.state_dict(),
        "config": cfg,
        "best_metrics": best_metrics,
    }, ckpt_path)
    (run_dir / "config.json").write_text(json.dumps(cfg, indent=2))
    (run_dir / "best_metrics.json").write_text(json.dumps(best_metrics, indent=2))
    return ckpt_path


In [ ]:

def run_one_config(lr: float, r: int, run_name: str, df_tr: pd.DataFrame, df_va: pd.DataFrame,
                   batch_size: int, grad_accum_steps: int):
    run_dir = OUT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    train_dl, val_dl = make_loaders(df_tr, df_va, batch_size=batch_size)

    model = WavLMWithHead(MODEL_NAME, r=r, lora_alpha=32, lora_dropout=0.05).to(DEVICE)
    tot, trn = count_params(model)
    print(f"[{run_name}] params: trainable {trn/1e6:.2f}M / total {tot/1e6:.2f}M")

    opt = torch.optim.AdamW(trainable_params(model), lr=lr, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler(enabled=(USE_AMP and DEVICE=="cuda"))

    best_S = -1e9
    best_epoch = 0
    best_ckpt = None
    best_metrics = None
    patience_cnt = 0

    history = []

    for epoch in range(1, MAX_EPOCH + 1):
        tr_loss = train_one_epoch(model, train_dl, opt, scaler, grad_accum_steps=grad_accum_steps)
        val_m = eval_metrics(model, val_dl)

        row = {"epoch": epoch, "train_mae": tr_loss, **{k: val_m[k] for k in ["mae_mean","acc_mean","rmse_mean","r2_mean","S"]}}
        history.append(row)

        print(
            f"[{run_name}] ep {epoch:02d} | "
            f"train_MAE {tr_loss:.4f} | "
            f"val_MAE {val_m['mae_mean']:.4f} | "
            f"val_(1-MAE) {val_m['acc_mean']:.4f} | "
            f"val_RMSE {val_m['rmse_mean']:.4f} | "
            f"val_R2 {val_m['r2_mean']:.4f} | "
            f"S {val_m['S']:.4f}"
        )

        if val_m["S"] > best_S:
            best_S = float(val_m["S"])
            best_epoch = int(epoch)
            best_metrics = {k: (val_m[k].tolist() if isinstance(val_m[k], np.ndarray) else val_m[k]) for k in val_m.keys()}
            patience_cnt = 0

            cfg = {
                "model_name": MODEL_NAME,
                "split": "strict",
                "lr": lr,
                "r": r,
                "seed": SEED,
                "metric_select": "S=1-MAE_mean",
                "max_epoch": MAX_EPOCH,
                "patience": PATIENCE,
                "weight_decay": WEIGHT_DECAY,
                "grad_clip": GRAD_CLIP,
                "batch_size": batch_size,
                "grad_accum_steps": grad_accum_steps,
                "use_amp": USE_AMP,
            }
            best_ckpt = save_checkpoint(run_dir, model, opt, epoch, best_S, cfg, best_metrics)
        else:
            patience_cnt += 1

        if patience_cnt >= PATIENCE:
            print(f"[{run_name}] Early stop (patience={PATIENCE}) at epoch {epoch}")
            break

    # save history
    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)

    return {
        "run_name": run_name,
        "lr": float(lr),
        "r": int(r),
        "batch_size": int(batch_size),
        "grad_accum_steps": int(grad_accum_steps),
        "best_S": float(best_S),
        "best_epoch": int(best_epoch),
        "best_mae": float(best_metrics["mae_mean"]) if best_metrics else None,
        "best_acc": float(best_metrics["acc_mean"]) if best_metrics else None,
        "best_rmse": float(best_metrics["rmse_mean"]) if best_metrics else None,
        "best_r2": float(best_metrics["r2_mean"]) if best_metrics else None,
        "ckpt": str(best_ckpt) if best_ckpt else None,
    }


In [ ]:
# ===== AUTO BATCH + QUICK SANITY (supaya gampang naikin kapasitas)
def _one_step_sanity(batch_size: int):
    # pakai subset kecil biar cepat
    df_tr_small = df_train.sample(min(len(df_train), 64), random_state=SEED)
    df_va_small = df_val.sample(min(len(df_val), 64), random_state=SEED)

    train_dl, val_dl = make_loaders(df_tr_small, df_va_small, batch_size=batch_size)

    model = WavLMWithHead(MODEL_NAME, r=8, lora_alpha=32, lora_dropout=0.05).to(DEVICE)
    opt = torch.optim.AdamW(trainable_params(model), lr=1e-4, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler(enabled=(USE_AMP and DEVICE=="cuda"))

    # 1 train step
    model.train()
    inputs, y = next(iter(train_dl))
    iv = inputs["input_values"].to(DEVICE, non_blocking=True)
    am = inputs.get("attention_mask", None)
    if am is not None: am = am.to(DEVICE, non_blocking=True)
    y = y.to(DEVICE, non_blocking=True)

    opt.zero_grad(set_to_none=True)
    with autocast(device_type=("cuda" if DEVICE=="cuda" else "cpu"), enabled=(USE_AMP and DEVICE=="cuda")):
        yhat = model(iv, am)
        loss = F.l1_loss(yhat, y)

    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(trainable_params(model), GRAD_CLIP)
    scaler.step(opt); scaler.update()

    # 1 val step (metrics)
    m = eval_metrics(model, val_dl)

    # cleanup
    del model, opt, scaler, train_dl, val_dl, inputs, y, iv, am, yhat, loss
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return m

def pick_max_batch_size(candidates):
    ok = []
    last_ok = None
    for bs in candidates:
        try:
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            _ = _one_step_sanity(bs)
            ok.append(bs)
            last_ok = bs
            print(f"✅ batch_size={bs} OK")
        except RuntimeError as e:
            msg = str(e).lower()
            if "out of memory" in msg or "cuda" in msg:
                print(f"❌ batch_size={bs} OOM/FAIL -> stop. ({e.__class__.__name__})")
                break
            raise
    return last_ok, ok

if AUTO_BATCH and DEVICE == "cuda":
    picked, ok_list = pick_max_batch_size(BS_CANDIDATES)
    if picked is not None:
        BATCH_SIZE = picked
        print("=> Picked BATCH_SIZE:", BATCH_SIZE, "| tried ok:", ok_list)
    else:
        print("=> AUTO_BATCH gagal, keep BATCH_SIZE =", BATCH_SIZE)

# grad accumulation (optional)
if TARGET_EFFECTIVE_BS is not None:
    GRAD_ACCUM_STEPS = max(1, math.ceil(TARGET_EFFECTIVE_BS / BATCH_SIZE))
else:
    GRAD_ACCUM_STEPS = 1

print("Final BATCH_SIZE =", BATCH_SIZE, "| GRAD_ACCUM_STEPS =", GRAD_ACCUM_STEPS)

# sanity metrics (pakai batch yang sudah dipilih)
m_sanity = _one_step_sanity(BATCH_SIZE)
print("Sanity metrics sample:",
      "MAE_mean", m_sanity["mae_mean"],
      "| (1-MAE)_mean", m_sanity["acc_mean"],
      "| RMSE_mean", m_sanity["rmse_mean"],
      "| R2_mean", m_sanity["r2_mean"])


In [ ]:
# ===== PHASE 1 — Tuning LR (r tetap)
R_DEFAULT = 8
LR_CANDIDATES = [1e-4, 2e-4]   # sesuai diagram (boleh tambah 5e-5 kalau mau)

phase1_results = []
for i, lr in enumerate(LR_CANDIDATES, start=1):
    run_name = f"phase1_run{i}_lr{lr:g}_r{R_DEFAULT}"
    res = run_one_config(lr=lr, r=R_DEFAULT, run_name=run_name,
                         df_tr=df_train, df_va=df_val,
                         batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS)
    phase1_results.append(res)

df_p1 = pd.DataFrame(phase1_results).sort_values("best_S", ascending=False)
display(df_p1)

LR_BEST = float(df_p1.iloc[0]["lr"])
print("LR_BEST:", LR_BEST)


In [ ]:
# ===== PHASE 2 — Tuning rank r (LR tetap)
R_CANDIDATES = [4, 16]  # sesuai diagram

phase2_results = []
for i, r in enumerate(R_CANDIDATES, start=1):
    run_name = f"phase2_run{i}_lr{LR_BEST:g}_r{r}"
    res = run_one_config(lr=LR_BEST, r=r, run_name=run_name,
                         df_tr=df_train, df_va=df_val,
                         batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS)
    phase2_results.append(res)

df_p2 = pd.DataFrame(phase2_results).sort_values("best_S", ascending=False)
display(df_p2)

R_BEST = int(df_p2.iloc[0]["r"])
BEST_EPOCH = int(df_p2.iloc[0]["best_epoch"])  # dipakai untuk retrain final tanpa tuning
print("R_BEST:", R_BEST, "| BEST_EPOCH (for final retrain):", BEST_EPOCH)


In [ ]:
# ===== RETRAIN FINAL (train_strict + val_strict) — tanpa tuning lagi
df_trainval = pd.concat([df_train, df_val], ignore_index=True)
trainval_ds = StrictAudioDataset(df_trainval)
trainval_dl = DataLoader(trainval_ds, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"),
                         persistent_workers=False, collate_fn=collate_fn)

final_dir = OUT_ROOT / f"final_retrain_lr{LR_BEST:g}_r{R_BEST}"
final_dir.mkdir(parents=True, exist_ok=True)

model_final = WavLMWithHead(MODEL_NAME, r=R_BEST, lora_alpha=32, lora_dropout=0.05).to(DEVICE)
opt_final = torch.optim.AdamW(trainable_params(model_final), lr=LR_BEST, weight_decay=WEIGHT_DECAY)
scaler_final = GradScaler(enabled=(USE_AMP and DEVICE=="cuda"))

print("Training final for epochs =", BEST_EPOCH)
for epoch in range(1, BEST_EPOCH + 1):
    tr_mae = train_one_epoch(model_final, trainval_dl, opt_final, scaler_final, grad_accum_steps=GRAD_ACCUM_STEPS)
    print(f"[FINAL] ep {epoch:02d} | train_MAE {tr_mae:.4f}")

# save final model
final_ckpt = final_dir / "final_model.pt"
cfg_final = {
    "model_name": MODEL_NAME,
    "split": "strict",
    "lr": LR_BEST,
    "r": R_BEST,
    "seed": SEED,
    "epochs": BEST_EPOCH,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "use_amp": USE_AMP,
}
torch.save({"model_state": model_final.state_dict(), "config": cfg_final}, final_ckpt)
(final_dir / "config.json").write_text(json.dumps(cfg_final, indent=2))

print("Saved:", final_ckpt)


In [ ]:
# ===== EVAL ON TEST (sekali, test_strict dikunci)
test_ds = StrictAudioDataset(df_test)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"),
                     persistent_workers=False, collate_fn=collate_fn)

# load final model
ck = torch.load(final_ckpt, map_location=DEVICE)
model_eval = WavLMWithHead(MODEL_NAME, r=R_BEST, lora_alpha=32, lora_dropout=0.05).to(DEVICE)
model_eval.load_state_dict(ck["model_state"], strict=True)

m_test = eval_metrics(model_eval, test_dl)

print("TEST strict:",
      "MAE_mean", m_test["mae_mean"],
      "| (1-MAE)_mean", m_test["acc_mean"],
      "| RMSE_mean", m_test["rmse_mean"],
      "| R2_mean", m_test["r2_mean"])

# save report
report = {
    "phase1": phase1_results,
    "phase2": phase2_results,
    "best": {"lr_best": LR_BEST, "r_best": R_BEST, "best_epoch_for_final": BEST_EPOCH},
    "final": {"final_ckpt": str(final_ckpt), "config": cfg_final},
    "test": {k: (m_test[k].tolist() if isinstance(m_test[k], np.ndarray) else m_test[k]) for k in m_test.keys()},
}
(final_dir / "report.json").write_text(json.dumps(report, indent=2))
pd.DataFrame(phase1_results + phase2_results).to_csv(OUT_ROOT / "all_runs_summary.csv", index=False)

print("Saved report:", final_dir / "report.json")
print("Saved summary:", OUT_ROOT / "all_runs_summary.csv")


### Notes (biar sesuai diagram)
- **Checkpoint tiap run** disimpan otomatis saat **S = 1−MAE_mean** membaik.
- **AUTO_BATCH** cuma buat sanity/kapasitas. Kalau kamu sudah yakin, boleh matikan dan set BATCH_SIZE manual.
- Kalau kamu naikkan batch size jauh, LR terbaik bisa geser sedikit. Tapi karena **Phase 1 memang tuning LR**, itu aman.
